# Lecture 7 — Class Exercise
## Heatmap & Waterfall: Netflix Catalogue

> **Push to:** `week07/lecture07_exercise.ipynb`

**Rules:**
1. Heatmap: colour scale must match the data type (sequential for counts, diverging for above/below)
2. Waterfall: use green for additions, red for subtractions, blue for totals
3. Insight title tells the setup-conflict-resolution story (or at minimum states the finding)
4. Annotate at least one cell or bar directly

---


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('/Users/sujith/Documents/Data_Visualization/data-viz-class-material/data/netflix_catalogue.csv')
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())


Loaded: 3000 titles
type
Movie      1974
TV Show    1026
Name: count, dtype: int64
      type  release_year  added_year             genre        country rating  \
0    Movie          2014        2016  Sci-Fi & Fantasy         France  PG-13   
1    Movie          2010        2014     Documentaries  United States  TV-MA   
2  TV Show          2011        2012     Kids & Family  United States  TV-14   
3    Movie          2016        2018             Anime          India     PG   
4    Movie          2014        2016     Kids & Family         Canada  TV-MA   

   duration  
0       157  
1       127  
2         6  
3       134  
4        77  


In [2]:
print("Genres:", df['genre'].value_counts().head(8))
print("\nCountries:", df['country'].value_counts().head(8))
print("\nRatings:", df['rating'].value_counts())


Genres: genre
Sports                244
Sci-Fi & Fantasy      213
Kids & Family         209
Crime                 206
Drama                 204
Horror                199
Action & Adventure    198
Thrillers             195
Name: count, dtype: int64

Countries: country
United States     932
India             337
United Kingdom    261
Japan             187
France            176
Canada            164
South Korea       151
Mexico            138
Name: count, dtype: int64

Ratings: rating
TV-MA    840
TV-14    733
PG-13    589
R        312
PG       196
TV-PG    128
G         92
TV-Y7     57
TV-G      53
Name: count, dtype: int64


## Task 1 — Heatmap: content by rating and release decade

**What to build:** A heatmap showing the number of titles by **content rating** (y-axis) and **decade** (x-axis).

**Requirements:**
- Create a 'decade' column: `df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'`
- Filter to TV-14, TV-MA, PG-13, R, PG only (most common ratings)
- Sequential colour scale (Blues)
- Values shown in cells (`text_auto=True`)
- Insight title about which rating dominates which decade


In [3]:
# Task 1
# YOUR CODE HERE

# Create decade column
df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'

# Keep common ratings
ratings = ['TV-MA', 'TV-14', 'PG-13', 'R', 'PG']
rating_df = df[df['rating'].isin(ratings)]

# Count titles by rating and decade
rating_decade = (
    rating_df.groupby(['rating', 'decade'])
    .size()
    .reset_index(name='count')
)

# Pivot table
pivot = rating_decade.pivot(
    index='rating',
    columns='decade',
    values='count'
).fillna(0)

# Sort rows by total count
pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]

# Build heatmap
fig = px.imshow(
    pivot,
    text_auto=True,
    color_continuous_scale='Blues',
    aspect='auto',
    labels={'color': 'Number of titles'},
    height=560,
    width=950
)

fig.update_traces(
    textfont=dict(size=12),
    hovertemplate='<b>%{y}</b><br>%{x}<br>Titles: %{z}<extra></extra>'
)

# Find highest cell
max_rating, max_decade = pivot.stack().idxmax()

# Add border around highest cell
x_labels = list(pivot.columns)
y_labels = list(pivot.index)

x_index = x_labels.index(max_decade)
y_index = y_labels.index(max_rating)

fig.add_shape(
    type='rect',
    xref='x',
    yref='y',
    x0=x_index - 0.5,
    x1=x_index + 0.5,
    y0=y_index - 0.5,
    y1=y_index + 0.5,
    line=dict(color='black', width=3),
    fillcolor='rgba(0,0,0,0)'
)

fig.update_layout(
    title=dict(
        text='TV-MA dominates Netflix’s recent catalogue, especially in the 2010s',
        x=0.02,
        xanchor='left'
    ),
    xaxis_title='Release Decade',
    yaxis_title='Content Rating',
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    margin=dict(l=80, r=40, t=70, b=60),
    coloraxis_colorbar=dict(title='Titles')
)

fig.update_xaxes(side='bottom')
fig.update_yaxes(autorange='reversed')

fig.show()

## Task 2 — Waterfall: Movie vs TV Show additions by year

**What to build:** A waterfall chart showing how Netflix's **Movie library** grew year by year (2015-2022).

**Requirements:**
- Filter to Movies only
- Group by `added_year`, count titles per year
- Final bar should be the cumulative total
- Green bars (additions), blue total
- Annotation on the year with the largest single addition
- Insight title naming the growth story


In [4]:
# Task 2
# YOUR CODE HERE

# Exploratory bar chart
adds = df.groupby('added_year').size().reset_index(name='new_titles')

fig = px.bar(
    adds,
    x='added_year',
    y='new_titles',
    title='Netflix Content Additions by Year',
    height=500
)

fig.show()

# Improved waterfall chart
adds = df.groupby('added_year').size().reset_index(name='new_titles')
adds = adds.loc[(adds['added_year'] >= 2015) & (adds['added_year'] <= 2022)].copy()
adds = adds.sort_values('added_year')

print(adds)

cumulative = adds['new_titles'].sum()

trace = go.Waterfall(
    x=adds['added_year'].astype(int).astype(str).tolist() + ['Total 2015-2022'],
    y=adds['new_titles'].tolist() + [cumulative],
    measure=['relative'] * len(adds) + ['total'],
    connector=dict(line=dict(color='#AAAAAA', dash='dot')),
    increasing=dict(marker_color='#70AD47'),
    totals=dict(marker_color='#2E75B6'),
    texttemplate='%{y:,}',
    textposition='outside'
)

fig = go.Figure(data=[trace])

fig.update_layout(
    title=dict(
    text='Netflix content additions remained consistently high from 2016 onward<br>with 2016 and 2019 showing the largest increases',
    x=0.02,
    xanchor='left'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    yaxis=dict(gridcolor='#EEEEEE', title='Titles Added'),
    xaxis=dict(title='Year', showgrid=False),
    margin=dict(l=60, r=40, t=55, b=40),
    showlegend=False,
    height=700
)

fig.show()

    added_year  new_titles
15        2015         111
16        2016         144
17        2017         121
18        2018         122
19        2019         129
20        2020         120
21        2021         125
22        2022         117
